In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bakehouse_bronze"
SILVER_SCHEMA = "bakehouse_silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

##Customers: dedupe on business key, drop missing keys

In [0]:
bronze_customers = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sales_customers")

silver_customers = (
    bronze_customers
    .dropDuplicates(["customerID"])
    .filter(F.col("customerID").isNotNull())
)

print(f"customers: {bronze_customers.count()} -> {silver_customers.count()}")

##Franchises: dedupe on business key, drop missing keys


In [0]:
bronze_franchises = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sales_franchises")

silver_franchises = (
    bronze_franchises
    .dropDuplicates(["franchiseID"])
    .filter(F.col("franchiseID").isNotNull())
)

print(f"franchises: {bronze_franchises.count()} -> {silver_franchises.count()}")

##Transactions: dedupe, validate, check referential integrity, mask PII

In [0]:
bronze_transactions = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_sales_transactions")
raw_count = bronze_transactions.count()

deduped = bronze_transactions.dropDuplicates(["transactionID"])
dupe_count = raw_count - deduped.count()

# basic validity: keys present, positive quantity/price
valid = deduped.filter(
    F.col("customerID").isNotNull() &
    F.col("franchiseID").isNotNull() &
    (F.col("quantity") > 0) &
    (F.col("unitPrice") > 0)
)
invalid_count = deduped.count() - valid.count()

# referential integrity: drop orphaned customer, franchise refs, but count them first
orphan_customer_count = valid.join(
    silver_customers.select("customerID"), "customerID", "left_anti"
).count()

orphan_franchise_count = valid.join(
    silver_franchises.select("franchiseID"), "franchiseID", "left_anti"
).count()

clean_refs = (
    valid
    .join(silver_customers.select("customerID"), "customerID", "left_semi")
    .join(silver_franchises.select("franchiseID"), "franchiseID", "left_semi")
)

# flag (not drop) totalPrice mismatches -- to decide gold layer logic later
silver_transactions = (
    clean_refs
    .withColumn("_price_mismatch", F.col("totalPrice") != (F.col("quantity") * F.col("unitPrice")))
    .withColumn("cardNumber_masked", F.concat(F.lit("****"), F.substring(F.col("cardNumber").cast("string"), -4, 4)))
    .drop("cardNumber")  # to never carry raw card numbers past bronze
)

mismatch_count = silver_transactions.filter(F.col("_price_mismatch")).count()

print(f"transactions: {raw_count} raw")
print(f"  duplicates removed: {dupe_count}")
print(f"  invalid (null keys / non-positive qty or price): {invalid_count}")
print(f"  orphaned customerID refs: {orphan_customer_count}")
print(f"  orphaned franchiseID refs: {orphan_franchise_count}")
print(f"  final silver row count: {silver_transactions.count()}")
print(f"  totalPrice/quantity*unitPrice mismatches flagged: {mismatch_count}")

##Write silver tables

In [0]:
silver_customers.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
silver_franchises.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_franchises")
silver_transactions.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.silver_transactions")

print("Silver layer written.")